In [4]:
pip install scapy pandas

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.6 MB 4.2 MB/s eta 0:00:01
   ---------------- ----------------------- 1.0/2.6 MB 5.0 MB/s eta 0:00:01
   ------------------------------------ --- 2.4/2.6 MB 5.4 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 4.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import numpy
import pandas
print(numpy.__version__)
print(pandas.__version__)

2.0.2
2.3.3


In [2]:
%pip install --upgrade --force-reinstall numpy pandas

  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl (15.9 MB)
   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.4 MB 3.7 MB/s eta 0:00:03
   --------- ------------------------------ 2.6/11.4 MB 6.9 MB/s eta 0:00:02
   -------------- ------------------------- 4.2/11.4 MB 6.8 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.4 MB 6.9 MB/s eta 0:00:01
   ------------------------ --------------- 7.1/11.4 MB 6.9 MB/s eta 0:00:01
   ------------------------------ --------- 8.7/11.4 MB 6.9 MB/s eta 0:00:01
   ------------------------------------ --- 10.2/11.4 MB 6.9 MB/s eta 0:00:01
   ---------------------------------------  11.3/11.4 MB 6.8 MB/s eta 0:00:01
   -----------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.


In [1]:
from scapy.all import rdpcap, TCP, UDP, IP
import pandas as pd

TCP_TIMEOUT = 60
UDP_TIMEOUT = 30

def normalize_tuple(proto, src, sport, dst, dport):
    if (src, sport) <= (dst, dport):
        return (proto, src, sport, dst, dport)
    else:
        return (proto, dst, dport, src, sport)

def extract_flows(pcap_file):
    packets = rdpcap(pcap_file)
    flows = {}

    for pkt in packets:
        if IP not in pkt:
            continue

        ip = pkt[IP]
        ts = pkt.time

        if TCP in pkt:
            l4 = pkt[TCP]
            proto = "TCP"
            timeout = TCP_TIMEOUT
        elif UDP in pkt:
            l4 = pkt[UDP]
            proto = "UDP"
            timeout = UDP_TIMEOUT
        else:
            continue

        key = normalize_tuple(
            proto,
            ip.src,
            l4.sport,
            ip.dst,
            l4.dport
        )

        if key not in flows:
            flows[key] = {
                "start_time": ts,
                "end_time": ts,
                "packet_count": 0,
                "byte_count": 0
            }

        # timeout handling
        if ts - flows[key]["end_time"] > timeout:
            flows[key]["start_time"] = ts
            flows[key]["packet_count"] = 0
            flows[key]["byte_count"] = 0

        flows[key]["end_time"] = ts
        flows[key]["packet_count"] += 1
        flows[key]["byte_count"] += len(pkt)

    records = []
    for i, (key, data) in enumerate(flows.items()):
        proto, src, sport, dst, dport = key
        records.append({
            "flow_id": i,
            "protocol": proto,
            "src_ip": src,
            "src_port": sport,
            "dst_ip": dst,
            "dst_port": dport,
            "start_time": data["start_time"],
            "end_time": data["end_time"],
            "duration": data["end_time"] - data["start_time"],
            "packet_count": data["packet_count"],
            "byte_count": data["byte_count"]
        })

    return pd.DataFrame(records)


In [ ]:
df = extract_flows("Bulk_only.pcap")
df.to_csv("flows_Bulk_raw.csv", index=False)

In [5]:
assert (df.packet_count > 0).all()
assert (df.duration >= 0).all()
assert (df.byte_count >= df.packet_count).all()